# deeplearning model 학습

## 1. 모델을 학습 모드로 설정 

- 모델의 train() 메소드를 호출하여 모델을 학습 모드로 설정, 
- 이는 일부레이어( dropout, batchnorm 등)에서 학습과 평가에서 다르게 동작해야 할 때 필요 

## 2. 데이터 배치 로드 

- DataLoader를 사용하여 학습 데이터셋으로부터 배치(batch)단위로 데이터를 로드,
- 각 배치는 모델의 입력으로 사용된다. 

## 3. 기울기 초기화 

- 옵티마이저의 zero_grad() 메소드를 호출하여 이전 스탭에서 계산된 기울기를 초기화 
- 이는 기울기가 누적되게 하지 않기 위함

## 4. 순전파 ( Foward Pass )

- 로드된 데이터 배치를 모델에 입력하여 순전파를 수행 
- 이 과정에서 모델은 데이터를 처리하고 예측 결과를 출력 

## 5. 손실 계산 

- 모델의 예측 결과와 실제 타켓(정답)을 비교하여 손실을 계산
- 미리 정의된 손실 함수를 사용

## 6. 역전파(Backward Pass)

- loss.backward()를 호출하여 손실에 대한 모델 파라미터들의 기울기를 계산. 
- 손실 함수의 기울기를 사용하여 파라미터 값들이 어떻게 변화해야 손실이 감소할 지 결정

## 7. 파라미터 업데이트 

- 옵티마이저의 step() 메소드를 호출하여 계산된 기울기를 사용해 모델 파라미터를 업데이트 
- 이 과정에서 학습률(learning rate)같은 하이퍼파라미터가 파라미터 업데이트에 영향을 미침 

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 더미 정형 데이터와 레이블 생성
num_samples = 1000  # 데이터 샘플의 수
num_test_samples = 200  # 테스트 데이터 샘플의 수
num_features = 10   # 입력 특성의 수


# 1. 랜덤 학습/테스트 데이터셋 생성
features = torch.randn(num_samples, num_features)
labels = (torch.rand(num_samples) > 0.5).long()  # 0과 1의 이진 레이블 생성

test_features = torch.randn(num_test_samples, num_features)
test_labels = (torch.rand(num_test_samples) > 0.5).long()  # 0과 1의 이진 레이블 생성


# 2. 학습/테스트 데이터셋과 데이터 로더 설정
dataset = TensorDataset(features, labels) # 데이터셋
train_loader = DataLoader(dataset, batch_size=64, shuffle=True) # 데이터 로더

test_dataset = TensorDataset(test_features, test_labels)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)  # 일반적으로 테스트 데이터를 섞을 필요는 없음

# 3. 신경망 모델 정의
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(num_features, 50)  # 입력층
        self.fc2 = nn.Linear(50, 20)            # 은닉층
        self.fc3 = nn.Linear(20, 2)             # 출력층

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

    
# 4. 모델, 손실 함수, 옵티마이저 초기화
model = SimpleNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [3]:
num_epochs = 10
model.train() # 모델을 학습 모드로 설정

for epoch in range(num_epochs):
    for inputs, targets in train_loader: # 데이터 로더
        optimizer.zero_grad()  # 기울기 초기화
        output = model(inputs)    # 순전파
        loss = criterion(output, targets)  # 손실 계산
        loss.backward()         # 역전파
        optimizer.step()        # 옵티마이저로 파라미터 업데이트

    print(f'Epoch {epoch+1}, Loss: {loss.item()}')

Epoch 1, Loss: 0.6861313581466675
Epoch 2, Loss: 0.6991921663284302
Epoch 3, Loss: 0.6950992941856384
Epoch 4, Loss: 0.6797186136245728
Epoch 5, Loss: 0.6882604360580444
Epoch 6, Loss: 0.6797246932983398
Epoch 7, Loss: 0.6780357956886292
Epoch 8, Loss: 0.6786917448043823
Epoch 9, Loss: 0.6715569496154785
Epoch 10, Loss: 0.6900526881217957


In [ ]:
# gradient가 실제로 채워지는지 확인
model.train()
inputs, targets = next(iter(train_loader))

# 1) backward 전: grad는 None 또는 0 상태
print("[Before backward]")
for name, p in model.named_parameters():
    print(name, p.grad is None)

# 2) 순전파 + 손실
outputs = model(inputs)
loss = criterion(outputs, targets)

# 3) 역전파
loss.backward()

print("\n[After backward] (gradient norm)")
for name, p in model.named_parameters():
    grad_norm = p.grad.norm().item() if p.grad is not None else None
    print(name, grad_norm)

# 4) grad 초기화
optimizer.zero_grad()
print("\n[After zero_grad]")
for name, p in model.named_parameters():
    if p.grad is None:
        print(name, "None")
    else:
        print(name, float(p.grad.abs().sum()))